# CSC413 - Project RNN Model Implementation

In [ ]:
import pandas as pd
import numpy as np
import nltk
import string
import re
import csv
import matplotlib.pyplot as plt
import nltk
nltk.download('stopwords')
from google.colab import files
!pip install -U torchtext==0.17.2

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Cleaning Process

In [ ]:
# From cleaning file (CSC413_Project_Data_Cleaning.ipynb) - only keeping nessessary functions

def convert_txt_to_csv(input_filename, output_filename):
    with open(input_filename, mode='r', newline='', encoding='utf-8') as infile, \
          open(output_filename, mode='w', newline='', encoding='utf-8') as outfile:

        reader = csv.reader(infile, delimiter=';')
        writer = csv.writer(outfile, delimiter=',')
        writer.writerow(['text', 'emotion'])
        for row in reader:
            writer.writerow(row)

traintxt = 'train.txt'
train = 'train.csv'
convert_txt_to_csv(traintxt, train)

testtxt = 'test.txt'
test = 'test.csv'
convert_txt_to_csv(testtxt, test)

valtxt = 'val.txt'
val = 'val.csv'
convert_txt_to_csv(valtxt, val)

def load_data(filename):
  df = pd.read_csv(filename)
  df['text'] = df['text'].str.lower().str.strip()
  df['emotion'] = df['emotion'].str.lower().str.strip()
  df = df.drop_duplicates()
  return df

def remove_punct(text):
  text  = "".join([c for c in text if c not in string.punctuation])
  text = re.sub('[0-9]+', '', text)
  return text

def remove_stopwords(text):
    stopword = nltk.corpus.stopwords.words('english')
    stopword.extend(
    ['yr', 'year', 'woman', 'man', 'girl','boy','one', 'two', 'sixteen',
     'yearold', 'fu', 'weeks', 'week', 'treatment', 'associated', 'patients',
     'may','day', 'case','old','u','n','didnt','ive','ate','feel','keep',
     'brother','dad','basic','im', 'feeling', 'felt'])
    keepword = [
    'not', 'no', 'nor', 'never', 'none', 'without', 'dont', 'didnt', 'doesnt',
    'isnt', 'arent', 'wasnt', 'werent', 'cant', 'couldnt', 'wont', 'wouldnt',
    'shouldnt', 'mustnt', 'havent', 'hasnt', 'hadnt', 'mightnt', 'shant', 'but']
    stopword = [word for word in stopword if word not in keepword]

    words = text.split()
    words = " ".join([w for w in words if w not in stopword])
    return words

def tokenize(text):
    words = text.split()
    return words

def clean_data(filename):
    if '.csv' in filename:
        df = load_data(filename)
        df['text'] = df['text'].apply(remove_punct)
        df['text'] = df['text'].apply(remove_stopwords)
        df['text'] = df['text'].apply(tokenize)
        return df
    else:
        print('Invalid File Type. Refer to convert_txt_to_csv, if applicable.')

In [ ]:
train = clean_data('train.csv')
val = clean_data('val.csv')
test = clean_data('test.csv')
labels = ['anger', 'fear', 'joy', 'love', 'sadness', 'surprise']

## Evaluation Metrics

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

def evaluate_metrics(y_pred, y_true=test['emotion'].values, labels=labels):
    print("Precision (per class):")
    print(precision_score(y_true, y_pred, average=None, labels=labels))

    print("\nRecall (per class):")
    print(recall_score(y_true, y_pred, average=None, labels=labels))

    print("\nF1-score (per class):")
    print(f1_score(y_true, y_pred, average=None, labels=labels))

    print("\nWeighted Precision:")
    print(precision_score(y_true, y_pred, average='weighted'))

    print("\nWeighted Recall:")
    print(recall_score(y_true, y_pred, average='weighted'))

    print("\nWeighted F1-score:")
    print(f1_score(y_true, y_pred, average='weighted'))

    print("\nFull Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=labels))

## Model Training

In [ ]:
import torch
from torchtext.vocab import build_vocab_from_iterator
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence


# Install torchtext if it's not already installed
try:
    import torchtext
except ImportError:
    !pip install torchtext

vocab = build_vocab_from_iterator(train['text'], specials=["<unk>", "<pad>"])
vocab.set_default_index(vocab["<unk>"])

print(f"Vocabulary size: {len(vocab)}")

label_map = {label: i for i, label in enumerate(labels)}
print(f"Label Map: {label_map}")


def collate_batch(batch):
    label_list, text_list = [], []

    for (_text, _label) in batch:
        # Convert label to integer
        label_list.append(label_map[_label])

        # Convert list of tokens to list of indices
        processed_text = torch.tensor([vocab[token] for token in _text], dtype=torch.int64)
        text_list.append(processed_text)


    label_list = torch.tensor(label_list, dtype=torch.int64)
    text_list = pad_sequence(text_list, padding_value=vocab['<pad>']).transpose(0, 1) # (batch_size, seq_len)

    return text_list, label_list


train_list = list(zip(train['text'], train['emotion']))
val_list = list(zip(val['text'], val['emotion']))
test_list = list(zip(test['text'], test['emotion']))

BATCH_SIZE = 64

train_dataloader = DataLoader(train_list, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
val_dataloader = DataLoader(val_list, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)
test_dataloader = DataLoader(test_list, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

print("DataLoaders created.")

Vocabulary size: 15042
Label Map: {'anger': 0, 'fear': 1, 'joy': 2, 'love': 3, 'sadness': 4, 'surprise': 5}
DataLoaders created.


In [ ]:
import torch.nn as nn

class EmotionRNN(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size, num_classes):
        super(EmotionRNN, self).__init__()
        self.emb = nn.Embedding(vocab_size, emb_size)
        self.rnn = nn.RNN(emb_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        embedded = self.emb(x)
        out, hidden = self.rnn(embedded)
        max_pool, _ = torch.max(out, dim=1)
        mean_pool = torch.mean(out, dim=1)

        cat_features = torch.cat((max_pool, mean_pool), dim=1)

        output = self.fc(cat_features)
        return output

VOCAB_SIZE = len(vocab)
EMB_SIZE = 100
HIDDEN_SIZE = 64
NUM_CLASSES = len(labels)

model = EmotionRNN(VOCAB_SIZE, EMB_SIZE, HIDDEN_SIZE, NUM_CLASSES)
print(model)

EmotionRNN(
  (emb): Embedding(15042, 100)
  (rnn): RNN(100, 64, batch_first=True)
  (fc): Linear(in_features=128, out_features=6, bias=True)
)


In [ ]:
import torch.optim as optim
import time

def train_model(model, train_loader, val_loader, num_epochs=10, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(num_epochs):
        start_time = time.time()
        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for text, labels in train_loader:
            optimizer.zero_grad()

            outputs = model(text)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            # Calculate accuracy
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_acc = correct / total
        avg_loss = total_loss / len(train_loader)

        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for text, labels in val_loader:
                outputs = model(text)
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_acc = val_correct / val_total

        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Loss: {avg_loss:.4f} | "
              f"Train Acc: {train_acc:.4f} | "
              f"Val Acc: {val_acc:.4f} | "
              f"Time: {time.time() - start_time:.2f}s")

# Train the model
train_model(model, train_dataloader, val_dataloader, num_epochs=10, lr=0.001)

Epoch 1/10 | Loss: 1.5472 | Train Acc: 0.3748 | Val Acc: 0.3685 | Time: 6.95s
Epoch 2/10 | Loss: 1.2689 | Train Acc: 0.5356 | Val Acc: 0.6140 | Time: 5.43s
Epoch 3/10 | Loss: 0.8648 | Train Acc: 0.7005 | Val Acc: 0.7455 | Time: 6.25s
Epoch 4/10 | Loss: 0.5290 | Train Acc: 0.8230 | Val Acc: 0.8150 | Time: 5.60s
Epoch 5/10 | Loss: 0.3360 | Train Acc: 0.8892 | Val Acc: 0.8515 | Time: 6.19s
Epoch 6/10 | Loss: 0.2272 | Train Acc: 0.9250 | Val Acc: 0.8560 | Time: 5.55s
Epoch 7/10 | Loss: 0.1638 | Train Acc: 0.9458 | Val Acc: 0.8660 | Time: 6.17s
Epoch 8/10 | Loss: 0.1203 | Train Acc: 0.9610 | Val Acc: 0.8690 | Time: 5.46s
Epoch 9/10 | Loss: 0.0920 | Train Acc: 0.9713 | Val Acc: 0.8710 | Time: 6.17s
Epoch 10/10 | Loss: 0.0692 | Train Acc: 0.9798 | Val Acc: 0.8755 | Time: 5.68s


## Hyperparameter Tuning

In [ ]:
def evaluate_hyperparams(train_data, val_data, vocab_size, num_classes,
                         emb_size, hidden_size, learning_rate, batch_size, num_epochs=5):

    print(f"Testing: Emb={emb_size}, Hidden={hidden_size}, LR={learning_rate}, Batch={batch_size}")

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)

    model = EmotionRNN(vocab_size, emb_size, hidden_size, num_classes)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(num_epochs):
        model.train()
        for X, t in train_loader:
            optimizer.zero_grad()
            output = model(X)
            loss = criterion(output, t)
            loss.backward()
            optimizer.step()

    val_acc = _get_accuracy(model, val_data, batch_size, collate_batch)
    print(f"  -> Validation Acc: {val_acc:.4f}")
    return val_acc

# Helper function to calculate accuracy
def _get_accuracy(model, data_list, batch_size, collate_fn):
    model.eval()
    correct = 0
    total = 0
    data_loader = DataLoader(data_list, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    with torch.no_grad():
        for text, labels in data_loader:
            outputs = model(text)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

embedding_sizes = [50, 100, 128]
hidden_sizes = [32, 64, 128]
learning_rates = [0.01, 0.001]
batch_sizes = [64]

best_acc = 0
best_config = {}


print("Starting Hyperparameter Tuning (5 epochs each)...")
print("-" * 60)

for emb in embedding_sizes:
    for hid in hidden_sizes:
        for lr in learning_rates:
            for batch in batch_sizes:
                acc = evaluate_hyperparams(
                    train_list,
                    val_list,
                    len(vocab),
                    len(labels),
                    emb, hid, lr, batch,
                    num_epochs=5
                )

                if acc > best_acc:
                    best_acc = acc
                    best_config = {
                        "emb_size": emb,
                        "hidden_size": hid,
                        "lr": lr,
                        "batch_size": batch
                    }

print("-" * 60)
print(f"Best Configuration Found:")
print(f"  Validation Accuracy: {best_acc:.4f}")
print(f"  Parameters: {best_config}")
print("-" * 60)

Starting Hyperparameter Tuning (5 epochs each)...
------------------------------------------------------------
Testing: Emb=50, Hidden=32, LR=0.01, Batch=64
  -> Validation Acc: 0.8870
Testing: Emb=50, Hidden=32, LR=0.001, Batch=64
  -> Validation Acc: 0.7205
Testing: Emb=50, Hidden=64, LR=0.01, Batch=64
  -> Validation Acc: 0.8950
Testing: Emb=50, Hidden=64, LR=0.001, Batch=64
  -> Validation Acc: 0.7705
Testing: Emb=50, Hidden=128, LR=0.01, Batch=64
  -> Validation Acc: 0.8080
Testing: Emb=50, Hidden=128, LR=0.001, Batch=64
  -> Validation Acc: 0.8270
Testing: Emb=100, Hidden=32, LR=0.01, Batch=64
  -> Validation Acc: 0.8755
Testing: Emb=100, Hidden=32, LR=0.001, Batch=64
  -> Validation Acc: 0.7715
Testing: Emb=100, Hidden=64, LR=0.01, Batch=64
  -> Validation Acc: 0.8995
Testing: Emb=100, Hidden=64, LR=0.001, Batch=64
  -> Validation Acc: 0.8515
Testing: Emb=100, Hidden=128, LR=0.01, Batch=64
  -> Validation Acc: 0.8785
Testing: Emb=100, Hidden=128, LR=0.001, Batch=64
  -> Validati

## Final Model

In [ ]:
best_emb_size = 50
best_hidden_size = 32
best_lr = 0.01
best_batch_size = 64
final_num_epochs = 15

print(f"Initializing Final Model with: Emb={best_emb_size}, Hidden={best_hidden_size}, LR={best_lr}, Batch={best_batch_size}")

final_model = EmotionRNN(vocab_size=len(vocab),
                    emb_size=best_emb_size,
                    hidden_size=best_hidden_size,
                    num_classes=len(labels))

train_dataloader_final = DataLoader(train_list, batch_size=best_batch_size, shuffle=True, collate_fn=collate_batch)
val_dataloader_final = DataLoader(val_list, batch_size=best_batch_size, shuffle=False, collate_fn=collate_batch)

train_model(final_model,
            train_dataloader_final,
            val_dataloader_final,
            num_epochs=final_num_epochs,
            lr=best_lr)

print("\nRunning Final Evaluation on Test Set...")
final_model.eval()

test_loader = DataLoader(test_list,
                         batch_size=best_batch_size,
                         shuffle=False,
                         collate_fn=collate_batch)

all_preds = []
all_labels = []

with torch.no_grad():
    for X, t in test_loader:
        z = final_model(X)
        preds = torch.argmax(z, dim=1)
        all_preds.extend(preds.tolist())
        all_labels.extend(t.tolist())

inv_label_map = {v: k for k, v in label_map.items()}
y_pred_str = [inv_label_map[p] for p in all_preds]
y_true_str = [inv_label_map[l] for l in all_labels]

evaluate_metrics(y_pred_str, y_true_str)


Initializing Final Model with: Emb=50, Hidden=32, LR=0.01, Batch=64
Epoch 1/15 | Loss: 1.1074 | Train Acc: 0.5762 | Val Acc: 0.8680 | Time: 3.81s
Epoch 2/15 | Loss: 0.2668 | Train Acc: 0.8976 | Val Acc: 0.8840 | Time: 3.19s
Epoch 3/15 | Loss: 0.1593 | Train Acc: 0.9337 | Val Acc: 0.8895 | Time: 3.19s
Epoch 4/15 | Loss: 0.1148 | Train Acc: 0.9522 | Val Acc: 0.8955 | Time: 3.86s
Epoch 5/15 | Loss: 0.0886 | Train Acc: 0.9662 | Val Acc: 0.8950 | Time: 3.20s
Epoch 6/15 | Loss: 0.0712 | Train Acc: 0.9721 | Val Acc: 0.8930 | Time: 3.18s
Epoch 7/15 | Loss: 0.0517 | Train Acc: 0.9813 | Val Acc: 0.8940 | Time: 3.30s
Epoch 8/15 | Loss: 0.0394 | Train Acc: 0.9862 | Val Acc: 0.8925 | Time: 3.83s
Epoch 9/15 | Loss: 0.0407 | Train Acc: 0.9849 | Val Acc: 0.8840 | Time: 3.16s
Epoch 10/15 | Loss: 0.0472 | Train Acc: 0.9822 | Val Acc: 0.8795 | Time: 3.18s
Epoch 11/15 | Loss: 0.0387 | Train Acc: 0.9856 | Val Acc: 0.8835 | Time: 3.52s
Epoch 12/15 | Loss: 0.0393 | Train Acc: 0.9857 | Val Acc: 0.8780 | Time: